# Contrastive learning: mechanism and RR experiment

This combines the synthetic Lightning demonstration with the RR contrastive study. Both use one configurable encoder, one NT-Xent objective, and one downstream classifier implementation.

Install the project first with `pip install -e ".[neural,signal,wavelet,notebooks]"`. Opening this notebook does not start training or download data.


In [ ]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
print(f"Project: {ROOT.name}")


## Synthetic mechanism demonstration

Two Gaussian clouds generate paired 3D points: a point and 1.1 times itself. This illustrates pair-based representation learning and is separate from ECG evaluation.


In [ ]:
from ecg_afib.datasets import run_synthetic_demo

RUN_DEMO = False
demo_root = ROOT / "runs/synthetic_demo"
if RUN_DEMO:
    run_synthetic_demo(demo_root, epochs=10, seed=42)
else:
    print("Synthetic training is disabled. Use ecg-afib demo to run it without ECG data.")


## Inspect learned synthetic coordinates

The shared embedding helper averages the two view embeddings per record. Labels color the plot but are not the pair IDs used in the contrastive loss.


In [ ]:
import json
import matplotlib.pyplot as plt

if (demo_root / "run/run.json").exists() and json.loads((demo_root / "run/run.json").read_text())["status"] == "complete":
    from ecg_afib.data import FeatureStore
    from ecg_afib.training import NeuralPredictor, record_embeddings
    from ecg_afib.splits import load_splits
    demo_store = FeatureStore(demo_root / "features")
    demo_splits = load_splits(demo_store, demo_root / "splits.json")
    predictor = NeuralPredictor.load(demo_root / "run")
    names, labels, latent = record_embeddings(predictor, demo_store, demo_splits["test"])
    fig, ax = plt.subplots(figsize=(5, 4))
    for label in (0, 1):
        ax.scatter(latent[labels == label, 0], latent[labels == label, 1], label=f"Synthetic class {label}")
    ax.set(xlabel="Embedding 1", ylabel="Embedding 2", title="Synthetic held-out representations")
    ax.legend()
    fig.tight_layout()
else:
    print("Run the synthetic demonstration to inspect its embeddings.")


## RR contrastive experiment

The real experiment uses two RR vectors from each training record. A frozen embedding feeds the supervised classifier. Classifier training, threshold selection, and test scoring use distinct record partitions.


In [ ]:
from ecg_afib.config import load_config
from ecg_afib.experiments import run_experiment

config = load_config(ROOT / "configs/rr_contrastive.toml")
RUN_RR_EXPERIMENT = False
if RUN_RR_EXPERIMENT:
    run_experiment(config)
else:
    print(config.to_dict())
